# Roxy report module tutorial

This notebook demonstrates how to use the **Roxy report module** to turn
EDA outputs into human-readable reports.

We will:

1. Generate a synthetic feature matrix and labels.
2. Build a `DatasetReport` using the EDA summary utilities.
3. Render the report as Markdown text.
4. Render the report as an HTML document.
5. Optionally save the reports to disk.


In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification

from roxy.features import (
    infer_feature_kinds,
    ColumnScaler,
)
from roxy.eda.summary import build_report
from roxy.report import (
    dataset_report_to_markdown,
    dataset_report_to_html,
)

## 1. Generate a simulated classification dataset

We use `sklearn.datasets.make_classification` to create a feature matrix
with 20 numeric features and a binary label. This mimics a feature block
produced by Roxy (e.g. `seq_global` descriptors).

In [2]:
X_num, y = make_classification(
    n_samples=400,
    n_features=20,
    n_informative=6,
    n_redundant=4,
    n_repeated=0,
    n_clusters_per_class=2,
    class_sep=1.5,
    flip_y=0.03,
    random_state=42,
)

feature_names = [f"f{i}" for i in range(1, X_num.shape[1] + 1)]
X = pd.DataFrame(X_num, columns=feature_names)
target = pd.Series(y, name="label")

X.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20
0,-0.856349,0.504582,-5.125331,1.700233,-3.012611,-0.745942,-3.591421,-0.614539,-0.611634,-3.757277,-0.346254,-1.049499,4.917199,-1.502871,-1.837191,3.492962,0.626538,-1.239562,0.753419,-0.661541
1,0.556553,0.257753,0.673417,0.353542,-1.902768,-1.907808,2.310563,1.887688,-1.335482,0.610092,-2.261808,-0.478163,0.939287,-0.155259,1.922556,2.566166,-1.241761,0.334176,-0.413606,-0.860385
2,-1.307370,-1.055089,-2.205620,-0.560278,1.359299,-0.423599,-0.841212,0.703428,-1.597872,0.787503,-2.507124,2.974150,1.098790,-0.792523,1.171275,-5.619259,0.126268,-0.984655,-0.974122,-1.365617
3,0.839664,-0.686279,0.975873,0.813138,-1.088523,0.514884,-1.875522,-0.532725,-1.357518,1.060243,0.728272,-0.131976,0.550930,0.585299,1.573537,-1.583697,-1.432671,0.145836,0.324340,0.819925
4,-0.073948,0.241322,3.484692,-0.000092,-0.922226,1.755476,2.691128,-0.150838,-0.451659,2.358294,-1.461856,-0.323512,-1.538259,-0.204471,3.865396,0.428105,-1.151014,0.386323,-0.465806,1.573020


## 2. Optional preprocessing and building a `DatasetReport`

In a typical Roxy workflow, you might:

1. Infer feature types (numeric, categorical, etc.).
2. Apply scaling to numeric features.
3. Build a `DatasetReport` using `build_report`, which summarises
   feature statistics, missingness, correlations and basic
   feature–target associations.

Here we perform a simple standardisation of numeric features and then
create the report.


In [3]:
# Infer feature types
kinds = infer_feature_kinds(X)
numeric_cols = [name for name, meta in kinds.items() if meta.kind.value == "numeric"]

numeric_cols

['f1',
 'f2',
 'f3',
 'f4',
 'f5',
 'f6',
 'f7',
 'f8',
 'f9',
 'f10',
 'f11',
 'f12',
 'f13',
 'f14',
 'f15',
 'f16',
 'f17',
 'f18',
 'f19',
 'f20']

In [4]:
# Scale numeric features (optional but typical before EDA)
scaler = ColumnScaler(strategy="standard", columns=numeric_cols)
X_scaled = scaler.fit_transform(X)

X_scaled.head()

,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17,f18,f19,f20
0,-0.802089,0.504289,-2.209755,1.824559,-1.469853,-0.767974,-1.776680,-0.560970,-0.538945,-1.892169,0.976797,-0.277980,1.729877,-1.686361,-1.411090,0.786993,0.649616,-1.247816,0.747717,-0.599310
1,0.607111,0.261905,0.395788,0.251705,-0.936108,-1.940316,1.095197,1.925855,-1.229714,0.295551,-0.602354,0.025234,0.094114,-0.205704,0.586305,0.567012,-1.106888,0.384107,-0.408573,-0.795657
2,-1.251929,-1.027296,-0.897846,-0.815583,0.632683,-0.442724,-0.438441,0.748884,-1.480114,0.384421,-0.804589,1.857409,0.159703,-0.905884,0.187181,-1.375851,0.179281,-0.983484,-0.963934,-1.294546
3,0.889481,-0.665127,0.531689,0.788485,-0.544522,0.504220,-0.941732,-0.479661,-1.250743,0.521043,1.862617,0.208959,-0.065583,0.607967,0.400886,-0.417984,-1.286375,0.188803,0.322585,0.863558
4,-0.021738,0.245770,1.658973,-0.161319,-0.464546,1.755998,1.280379,-0.100123,-0.386282,1.171268,0.057113,0.107308,-0.924682,-0.259775,1.618455,0.059530,-1.021572,0.438181,-0.460293,1.607198


In [5]:
# Build a dataset-level EDA report
dataset_report = build_report(
    X_scaled,
    y=target,
    dataset_name="Simulated classification dataset",
    task_type="classification",
)

dataset_report

DatasetReport(dataset_name='Simulated classification dataset', n_samples=400, n_features=20, task_type='classification', class_distribution={0: 200, 1: 200}, feature_summaries={'f1': FeatureSummary(name='f1', dtype='float64', n_missing=0, missing_ratio=0.0, n_unique=400, mean=4.440892098500626e-18, std=1.0012523486435179, min=-2.8697583174031505, max=2.9799838278809356, skewness=-0.06253529509671925, kurtosis=0.0024178100811402814, target_association={'feature_name': 'f1', 'target_name': 'label', 'test_name': 'kruskal', 'statistic': 0.9334331670822849, 'p_value': 0.3339724049113174, 'n_groups': 2, 'group_sizes': {0: 200, 1: 200}, 'effect_size': -0.0001672533490394853, 'posthoc_pvalues': None, 'notes': ['Effect size is epsilon-squared for Kruskal–Wallis.']}), 'f2': FeatureSummary(name='f2', dtype='float64', n_missing=0, missing_ratio=0.0, n_unique=400, mean=-4.440892098500626e-18, std=1.001252348643518, min=-2.6468972028958766, max=3.1934852479262155, skewness=0.2062408831129404, kurtos

## 3. Rendering the report as Markdown

We can convert the `DatasetReport` into a Markdown document using
`dataset_report_to_markdown`. This is useful for:

- embedding in Jupyter notebooks,
- saving alongside experiments,
- version-controlling reports in text form,
- converting to HTML/PDF with external tools.


In [6]:
md_report = dataset_report_to_markdown(
    dataset_report,
    max_features=30,        # limit number of features in the table, if desired
    include_correlation=True,
    float_fmt=".3f",
)

# Show the first 60 lines of the Markdown report for inspection
md_report.splitlines()[:60]

/home/dmedina/Desktop/tools_base/roxy_library/roxy/report/markdown.py:194: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df[float_cols] = df[float_cols].applymap(


['# Roxy dataset report — Simulated classification dataset',
 '',
 '## Overview',
 '',
 '- **Samples:** `400`',
 '- **Features:** `20`',
 '- **Task type:** `classification`',
 '- **Class distribution:**',
 '  - `0`: `200`',
 '  - `1`: `200`',
 '',
 '## Feature summary',
 '',
 '| name | dtype | n_missing | missing_ratio | n_unique | mean | std | min | max | skewness | kurtosis |',
 '| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |',
 '| f1 | float64 | 0 | 0.000 | 400 | 0.000 | 1.001 | -2.870 | 2.980 | -0.063 | 0.002 |',
 '| f2 | float64 | 0 | 0.000 | 400 | -0.000 | 1.001 | -2.647 | 3.193 | 0.206 | 0.094 |',
 '| f3 | float64 | 0 | 0.000 | 400 | -0.000 | 1.001 | -2.426 | 2.931 | 0.290 | 0.181 |',
 '| f4 | float64 | 0 | 0.000 | 400 | 0.000 | 1.001 | -2.605 | 3.011 | 0.137 | 0.048 |',
 '| f5 | float64 | 0 | 0.000 | 400 | 0.000 | 1.001 | -2.462 | 2.160 | -0.146 | -0.688 |',
 '| f6 | float64 | 0 | 0.000 | 400 | 0.000 | 1.001 | -2.664 | 3.089 | 0.078 | -0.125 |',
 '| f7 | fl

## 4. Rendering the report as HTML

We can also convert the same `DatasetReport` into a standalone HTML
document using `dataset_report_to_html`.

This is handy for:

- sharing reports outside of notebooks,
- embedding inside dashboards and web applications,
- exporting static documentation of datasets.


In [7]:
html_report = dataset_report_to_html(
    dataset_report,
    max_features=30,
    include_correlation=True,
)

# Print the first ~50 lines for a quick preview
html_report.splitlines()[:50]

['<!DOCTYPE html>',
 "<html lang='en'>",
 '<head>',
 "<meta charset='utf-8' />",
 '<title>Simulated classification dataset</title>',
 '<style>',
 'body {',
 '  font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;',
 '  margin: 2rem;',
 '  line-height: 1.5;',
 '}',
 'h1, h2 {',
 '  color: #222;',
 '}',
 'code {',
 '  font-family: "Fira Code", Menlo, Consolas, monospace;',
 '  background-color: #f5f5f5;',
 '  padding: 0.1rem 0.25rem;',
 '  border-radius: 3px;',
 '}',
 '.table {',
 '  border-collapse: collapse;',
 '  width: 100%;',
 '  margin-bottom: 1.5rem;',
 '}',
 '.table th,',
 '.table td {',
 '  border: 1px solid #ddd;',
 '  padding: 0.35rem 0.5rem;',
 '  font-size: 0.9rem;',
 '}',
 '.table-striped tbody tr:nth-child(odd) {',
 '  background-color: #fafafa;',
 '}',
 '.caption {',
 '  font-size: 0.9rem;',
 '  color: #555;',
 '}',
 '.overview ul {',
 '  list-style: disc;',
 '}',
 '</style>',
 '</head>',
 '<body>',
 '<h1>Roxy dataset report — Simulated class

In [8]:
# If running inside a notebook that supports HTML display,
# we can render the HTML directly for visual inspection.

from IPython.display import HTML

HTML(html_report)

name,dtype,n_missing,missing_ratio,n_unique,mean,std,min,max,skewness,kurtosis
f1,float64,0,0.0,400,0.0,1.001,-2.870,2.980,-0.063,0.002
f2,float64,0,0.0,400,-0.0,1.001,-2.647,3.193,0.206,0.094
f3,float64,0,0.0,400,-0.0,1.001,-2.426,2.931,0.290,0.181
f4,float64,0,0.0,400,0.0,1.001,-2.605,3.011,0.137,0.048
f5,float64,0,0.0,400,0.0,1.001,-2.462,2.160,-0.146,-0.688
f6,float64,0,0.0,400,0.0,1.001,-2.664,3.089,0.078,-0.125
f7,float64,0,0.0,400,0.0,1.001,-2.314,2.338,0.040,-0.821
f8,float64,0,0.0,400,0.0,1.001,-2.772,2.857,-0.096,-0.306
f9,float64,0,0.0,400,-0.0,1.001,-2.981,3.792,0.051,0.160
f10,float64,0,0.0,400,-0.0,1.001,-2.575,2.982,0.058,-0.287


## 5. Saving the reports to disk

Finally, we may want to persist the Markdown and HTML reports as files.
This mirrors how you might archive dataset documentation in a project
folder or pipeline artefact store.


In [9]:
# Define output paths (relative to current working directory)
md_path = "roxy_dataset_report.md"
html_path = "roxy_dataset_report.html"

with open(md_path, "w", encoding="utf-8") as f:
    f.write(md_report)

with open(html_path, "w", encoding="utf-8") as f:
    f.write(html_report)

md_path, html_path

('roxy_dataset_report.md', 'roxy_dataset_report.html')

## 6. Summary

In this notebook we have:

- Created a synthetic dataset representative of a Roxy feature block.
- Built a `DatasetReport` using the EDA summary tools.
- Rendered the report as Markdown for text-based inspection and
  version control.
- Rendered the report as HTML for sharing and embedding in front-ends.
- Optionally saved both representations to disk.

These patterns can be applied directly to real `RoxyDataset` objects,
allowing automated, reproducible dataset documentation across your
machine learning workflows.
